# 00 · Overview and setup

**What this folder does.** Adds two comparison datasets to the BabyView (BV) analyses: **ImageNet-1k** (whole
images, one label each) and **LVIS v1** (human-labeled object boxes, cropped). Each dataset's classes are mapped
onto CDI words by WordNet hyponym pooling, category embeddings are built the way the manuscript builds THINGS, and
BV is compared with them, with THINGS and VEDB alongside.

**The notebooks are the source code.** Each one is self-contained: it defines every function it uses, so you can
edit any cell, rerun, and save. Notebooks hand results to each other only through files in `../outputs/`. Shared
helpers (paths, table loading) are deliberately repeated in each notebook so none depends on another's code.

| # | Notebook | Dataset parameter | Reads | Writes |
|---|---|---|---|---|
| 01 | CDI mapping (WordNet hyponym pooling) | `imagenet` or `lvis` | images / annotations | `mapping/` maps, anchors, LVIS crop manifest |
| 02 | Mapping audit | both | 01, 05 | `mapping/mapping_audit_*`, review page |
| 03 | Class / category mean embeddings | `imagenet` or `lvis` | embeddings (ccn2) | `mapping/*_embeddings_*.npz`, LVIS crop sizes |
| 04 | Exemplar montages | `imagenet` or `lvis` | 01 | `montages/` |
| 05 | Stage 0 category tables | `imagenet` or `lvis` | 01, 03, manuscript tables | `exemplar_sets/` |
| 06 | BV vs THINGS vs comparison dataset | `imagenet` or `lvis` | 05 | `bv_<dataset>/` |
| 07 | RDMs clustered within CDI domains | any 2+ of bv vedb things imagenet lvis | 05 | `rdms/<datasets>/` |
| 08 | RDM correlation matrices | any 2+ | 05 | `rdm_matrices/<datasets>/` |
| 09 | CDI-domain cluster strength | any 2+ | 05 | `cluster_strength/<datasets>/` |
| 10 | Category prevalence | any of bv vedb imagenet lvis | 01, 05, BV/VEDB tables | `category_prevalence/` |

Run order: 01 (both datasets) → 03 → 05 (both) → 02 → the rest in any order. GPU embedding runs live in
`../data_prep/` (shell scripts). **Kernel:** *Python (vislearnlabpy)*, the environment every result was made with.

This notebook checks that every input exists.

In [1]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

plt.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [2]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

In [3]:
# Dataset registry and raw-table loading (shared helpers; repeated in each notebook that needs them).
ALL_DATASETS = ("bv", "vedb", "things", "imagenet", "lvis")
LABEL = {"bv": "BV", "vedb": "VEDB", "things": "THINGS", "imagenet": "ImageNet", "lvis": "LVIS"}
COLOR = {"bv": "#90caf9", "vedb": "#0d47a1", "imagenet": "#eb6834", "lvis": "#1baf7a", "things": "#eda100"}
COMPARISON_SETS = {"imagenet": "imagenet_exclusive75", "lvis": "lvis_exclusive104"}


def zscore_rows(X: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """Same as manuscript notebook 07: z-score each dimension across categories."""
    return (X - X.mean(axis=0)) / (X.std(axis=0) + eps)


def raw_path(ds: str, model: str) -> Path:
    if ds in COMPARISON_SETS:
        s = COMPARISON_SETS[ds]
        return EXEMPLAR_SETS / s / f"{ds}_{model}_exemplar_avg_raw_within_{s}.csv"
    if ds == "vedb":
        return VEDB_VALID129 / f"bv_{model}_exemplar_avg_raw_within_valid129.csv"
    return BV_THINGS_VALID129 / f"{ds}_{model}_exemplar_avg_raw_within_valid129.csv"


def load_raw(path: Path, cats: list[str]) -> np.ndarray:
    df = pd.read_csv(path)
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.set_index("category").loc[cats].filter(like="dim_").to_numpy(dtype=np.float64)


def shared_categories(datasets) -> list[str]:
    """Categories present in every given dataset's table, for both models (sorted)."""
    present = None
    for ds in datasets:
        for model in MODELS:
            cats = set(pd.read_csv(raw_path(ds, model), usecols=["category"])["category"].str.strip().str.lower())
            present = cats if present is None else present & cats
    return sorted(present)


def zscored_tables(datasets, cats, model) -> dict[str, np.ndarray]:
    """{dataset: category x dim matrix}, each z-scored within cats."""
    return {ds: zscore_rows(load_raw(raw_path(ds, model), cats)) for ds in datasets}

## Inputs present?

In [4]:
checks = {
    "BV / THINGS valid129 tables (manuscript Stage 0)": BV_THINGS_VALID129,
    "VEDB valid129 tables (vedb-1612 Stage 0)": VEDB_VALID129,
    "ImageNet images (/data2)": IMAGENET_DIR, "ImageNet embeddings (/ccn2a)": IMAGENET_EMBED_ROOT,
    "LVIS annotations + images (/data2)": LVIS_ROOT, "LVIS crop embeddings (/ccn2a)": LVIS_CROP_EMBED_ROOT,
    "CDI word list": VOCAB_DIR / "cdi_concrete_nouns_295_with_semantic.csv",
}
for ds in ALL_DATASETS:
    for model in MODELS:
        checks[f"category table: {ds} {model}"] = raw_path(ds, model)
pd.DataFrame([{"input": k, "path": str(v), "exists": Path(v).exists()} for k, v in checks.items()])

,input,path,exists
0,BV / THINGS valid129 tables (manuscript Stage 0),/home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129,True
1,VEDB valid129 tables (vedb-1612 Stage 0),/home/j7yang/babyview-projects/vss2026/object-detection/analysis/vedb-1612/exemplar_set_embeddings/valid129,True
2,ImageNet images (/data2),/data2/j7yang/imagenet-1k/extracted,True
3,ImageNet embeddings (/ccn2a),/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings,True
4,LVIS annotations + images (/data2),/data2/j7yang/lvis,True
5,LVIS crop embeddings (/ccn2a),/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings,True
6,CDI word list,/home/j7yang/babyview-projects/vss2026/object-detection/data/vocabulary_lists/cdi_concrete_nouns_295_with_semantic.csv,True
7,category table: bv clip,/home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129/bv...,True
8,category table: bv dinov3,/home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129/bv...,True
9,category table: vedb clip,/home/j7yang/babyview-projects/vss2026/object-detection/analysis/vedb-1612/exemplar_set_embeddings/valid129/bv_clip_...,True
